# Аномалии ИНН за февраль (Excel vs ODS SA-active)

Проверяем правила:
1. У каждого уникального ИНН должен быть один договор.
2. Количество уникальных ИНН должно совпадать с количеством уникальных наименований.
3. В ODS должен быть только тип договора `SA` (контроль аномалий по `acq_class`).
4. Проверка кейса смены договора в расчетном периоде (закрытие + открытие в феврале).

Источники:
- Excel: `02_Февраль_2026.xlsx`
- ODS: `ods_alpha.scd1_agreements` + `ods_alpha.scd1_companies`.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
month_start = '2026-02-01'

excel_inn_col = 'ИНН'
excel_contract_col = 'Номер договора'
excel_name_col = 'Наименование'

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

def normalize_text(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def detect_excel_header_row(path, required_cols, scan_rows=30):
    raw = pd.read_excel(path, header=None)
    req = set([str(c).strip() for c in required_cols])
    max_rows = min(scan_rows, len(raw))
    for i in range(max_rows):
        vals = set([str(x).strip() for x in raw.iloc[i].tolist()])
        if len(req & vals) >= 2:
            return i
    return 0

## 0) Загрузка данных Excel и ODS в единый формат

In [ ]:
# Excel
header_row = detect_excel_header_row(excel_path, [excel_inn_col, excel_contract_col, excel_name_col])
df_excel_raw = pd.read_excel(excel_path, header=header_row)
df_excel_raw.columns = [str(c).strip() for c in df_excel_raw.columns]

for c in [excel_inn_col, excel_contract_col, excel_name_col]:
    if c not in df_excel_raw.columns:
        raise ValueError(f'В Excel отсутствует колонка: {c}')

excel_df = pd.DataFrame({
    'inn': df_excel_raw[excel_inn_col].apply(normalize_id),
    'contract_number': df_excel_raw[excel_contract_col].apply(normalize_text),
    'client_name': df_excel_raw[excel_name_col].apply(normalize_text),
})
excel_df['acq_class'] = None
excel_df['d_valid_from'] = None
excel_df['d_valid_to'] = None
excel_df['source'] = 'excel'
excel_df = excel_df.dropna(subset=['inn']).drop_duplicates()

# ODS (весь договорный слой для диагностики дат/типов) + отдельный SA-active срез
with imp:
    ods_all_raw = imp.fetch(f"""
        select
            cast(c.c_inn as string) as inn,
            cast(a.c_agr_number as string) as contract_number,
            cast(c.c_cmp_name as string) as client_name,
            cast(a.acq_class as string) as acq_class,
            cast(a.d_valid_from as date) as d_valid_from,
            cast(a.d_valid_to as date) as d_valid_to
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where c.c_inn is not null
    """)

    ods_sa_active_raw = imp.fetch(f"""
        select
            cast(c.c_inn as string) as inn,
            cast(a.c_agr_number as string) as contract_number,
            cast(c.c_cmp_name as string) as client_name,
            cast(a.acq_class as string) as acq_class,
            cast(a.d_valid_from as date) as d_valid_from,
            cast(a.d_valid_to as date) as d_valid_to
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where a.acq_class = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and c.c_inn is not null
    """)

ods_all_df = pd.DataFrame({
    'inn': ods_all_raw['inn'].apply(normalize_id),
    'contract_number': ods_all_raw['contract_number'].apply(normalize_text),
    'client_name': ods_all_raw['client_name'].apply(normalize_text),
    'acq_class': ods_all_raw['acq_class'].apply(normalize_text),
    'd_valid_from': pd.to_datetime(ods_all_raw['d_valid_from'], errors='coerce').dt.date,
    'd_valid_to': pd.to_datetime(ods_all_raw['d_valid_to'], errors='coerce').dt.date,
})
ods_all_df['source'] = 'ods_all'
ods_all_df = ods_all_df.dropna(subset=['inn']).drop_duplicates()

ods_df = pd.DataFrame({
    'inn': ods_sa_active_raw['inn'].apply(normalize_id),
    'contract_number': ods_sa_active_raw['contract_number'].apply(normalize_text),
    'client_name': ods_sa_active_raw['client_name'].apply(normalize_text),
    'acq_class': ods_sa_active_raw['acq_class'].apply(normalize_text),
    'd_valid_from': pd.to_datetime(ods_sa_active_raw['d_valid_from'], errors='coerce').dt.date,
    'd_valid_to': pd.to_datetime(ods_sa_active_raw['d_valid_to'], errors='coerce').dt.date,
})
ods_df['source'] = 'ods_sa_active'
ods_df = ods_df.dropna(subset=['inn']).drop_duplicates()

display(pd.DataFrame([
    {'source': 'excel', 'header_row': header_row, 'rows': len(excel_df), 'unique_inn': excel_df['inn'].nunique()},
    {'source': 'ods_sa_active', 'header_row': None, 'rows': len(ods_df), 'unique_inn': ods_df['inn'].nunique()},
    {'source': 'ods_all', 'header_row': None, 'rows': len(ods_all_df), 'unique_inn': ods_all_df['inn'].nunique()},
]))

## 1) Правило: у одного ИНН должен быть один договор

In [ ]:
def rule1_anomalies(df, source_name):
    z = df.dropna(subset=['inn']).copy()
    z['contract_key'] = z['contract_number'].fillna('__NULL__')
    g = (
        z.groupby('inn', as_index=False)
         .agg(contract_cnt=('contract_key', 'nunique'))
    )
    bad = g[g['contract_cnt'] > 1].copy()
    details = z[z['inn'].isin(bad['inn'])][['inn', 'contract_number', 'client_name']].drop_duplicates()
    summary = pd.DataFrame([{
        'source': source_name,
        'unique_inn': z['inn'].nunique(),
        'inn_with_multi_contracts': bad['inn'].nunique(),
        'share_pct': round(100.0 * bad['inn'].nunique() / max(z['inn'].nunique(), 1), 2)
    }])
    return summary, bad, details

r1_excel_summary, r1_excel_bad, r1_excel_details = rule1_anomalies(excel_df, 'excel')
r1_ods_summary, r1_ods_bad, r1_ods_details = rule1_anomalies(ods_df, 'ods_sa_active')

display(pd.concat([r1_excel_summary, r1_ods_summary], ignore_index=True))
print('Excel anomalies (top 100):')
display(r1_excel_details.head(100))
print('ODS anomalies (top 100):')
display(r1_ods_details.head(100))

## 2) Правило: количество уникальных ИНН = количеству уникальных наименований

In [ ]:
def rule2_checks(df, source_name):
    z = df.copy()
    unique_inn = z['inn'].dropna().nunique()
    unique_name = z['client_name'].dropna().nunique()

    inn_to_names = (
        z.dropna(subset=['inn', 'client_name'])
         .groupby('inn', as_index=False)
         .agg(name_cnt=('client_name', 'nunique'))
    )
    inn_multi_names = inn_to_names[inn_to_names['name_cnt'] > 1]

    name_to_inn = (
        z.dropna(subset=['inn', 'client_name'])
         .groupby('client_name', as_index=False)
         .agg(inn_cnt=('inn', 'nunique'))
    )
    name_multi_inn = name_to_inn[name_to_inn['inn_cnt'] > 1]

    summary = pd.DataFrame([{
        'source': source_name,
        'unique_inn': unique_inn,
        'unique_client_name': unique_name,
        'delta_inn_minus_name': unique_inn - unique_name,
        'inn_with_multi_names': len(inn_multi_names),
        'names_with_multi_inn': len(name_multi_inn),
    }])

    inn_multi_names_details = z[z['inn'].isin(inn_multi_names['inn'])][['inn', 'client_name', 'contract_number']].drop_duplicates()
    name_multi_inn_details = z[z['client_name'].isin(name_multi_inn['client_name'])][['client_name', 'inn', 'contract_number']].drop_duplicates()
    return summary, inn_multi_names_details, name_multi_inn_details

r2_excel_summary, r2_excel_inn_names, r2_excel_name_inn = rule2_checks(excel_df, 'excel')
r2_ods_summary, r2_ods_inn_names, r2_ods_name_inn = rule2_checks(ods_df, 'ods_sa_active')

display(pd.concat([r2_excel_summary, r2_ods_summary], ignore_index=True))
print('Excel: INN with multiple names (top 100):')
display(r2_excel_inn_names.head(100))
print('Excel: names with multiple INN (top 100):')
display(r2_excel_name_inn.head(100))
print('ODS: INN with multiple names (top 100):')
display(r2_ods_inn_names.head(100))
print('ODS: names with multiple INN (top 100):')
display(r2_ods_name_inn.head(100))

## 3) Финальная сводка нарушений и top-примеры

In [ ]:
# Доп. проверка: тип договора в ODS (должен быть только SA в SA-active срезе)
acq_class_check = (
    ods_df.groupby('acq_class', dropna=False, as_index=False)
          .agg(rows=('inn', 'count'), unique_inn=('inn', 'nunique'))
)

# Доп. проверка: кейс смены договора в феврале (закрытие и открытие нового в расчетном месяце)
month_start_dt = pd.to_datetime(month_start).date()
month_end_dt = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).date()

ods_all_work = ods_all_df.copy()
ods_all_work['d_valid_from'] = pd.to_datetime(ods_all_work['d_valid_from'], errors='coerce').dt.date
ods_all_work['d_valid_to'] = pd.to_datetime(ods_all_work['d_valid_to'], errors='coerce').dt.date

closed_in_feb = ods_all_work[
    ods_all_work['d_valid_to'].notna() &
    (ods_all_work['d_valid_to'] >= month_start_dt) &
    (ods_all_work['d_valid_to'] <= month_end_dt)
][['inn', 'contract_number', 'd_valid_to']].drop_duplicates()

opened_in_feb = ods_all_work[
    ods_all_work['d_valid_from'].notna() &
    (ods_all_work['d_valid_from'] >= month_start_dt) &
    (ods_all_work['d_valid_from'] <= month_end_dt)
][['inn', 'contract_number', 'd_valid_from']].drop_duplicates()

contract_switch_feb = (
    closed_in_feb.merge(opened_in_feb[['inn', 'contract_number', 'd_valid_from']], on='inn', how='inner', suffixes=('_closed', '_opened'))
)
contract_switch_feb = contract_switch_feb[contract_switch_feb['contract_number_closed'] != contract_switch_feb['contract_number_opened']]

switch_inn_cnt = contract_switch_feb['inn'].nunique() if len(contract_switch_feb) else 0

final_summary = pd.concat([
    r1_excel_summary.assign(rule='rule1_inn_one_contract'),
    r1_ods_summary.assign(rule='rule1_inn_one_contract'),
    r2_excel_summary.assign(rule='rule2_unique_inn_eq_unique_name'),
    r2_ods_summary.assign(rule='rule2_unique_inn_eq_unique_name'),
], ignore_index=True, sort=False)

final_extra = pd.DataFrame([
    {
        'rule': 'rule3_acq_class_only_sa_in_ods_sa_active',
        'source': 'ods_sa_active',
        'unique_inn': ods_df['inn'].nunique(),
        'unique_client_name': None,
        'delta_inn_minus_name': None,
        'inn_with_multi_names': None,
        'names_with_multi_inn': None,
        'inn_with_multi_contracts': None,
        'share_pct': None,
        'note': 'В ods_sa_active должны быть только acq_class = SA'
    },
    {
        'rule': 'rule4_contract_switch_within_feb',
        'source': 'ods_all',
        'unique_inn': switch_inn_cnt,
        'unique_client_name': None,
        'delta_inn_minus_name': None,
        'inn_with_multi_names': None,
        'names_with_multi_inn': None,
        'inn_with_multi_contracts': None,
        'share_pct': None,
        'note': 'ИНН с закрытием и открытием другого договора в феврале'
    }
])

display(final_summary)
print('acq_class distribution in ods_sa_active:')
display(acq_class_check)
print('contract switch within February (top 100):')
display(contract_switch_feb.head(100))

display(final_extra)

print('TOP examples | Excel rule1:')
display(r1_excel_details.head(30))
print('TOP examples | ODS rule1:')
display(r1_ods_details.head(30))
print('TOP examples | Excel rule2 (inn -> names):')
display(r2_excel_inn_names.head(30))
print('TOP examples | ODS rule2 (inn -> names):')
display(r2_ods_inn_names.head(30))

## 4) Диагностика кейсов, где `client_name` выглядит как РФ/филиал

Блок ищет в ODS клиентов, у которых `client_name` похож на организационную сущность (например, `РФ`, `филиал`, `отделение`), и показывает:
- масштаб проблемы;
- примеры строк из ODS;
- сверку тех же ИНН с Excel.

In [ ]:
suspect_patterns = [
    r'(^|\s)рф($|\s)',
    r'филиал',
    r'отделени',
    r'доп\.?\s*офис',
    r'территориаль',
]
pattern_union = '|'.join(suspect_patterns)

ods_name_flags = ods_all_df.copy()
ods_name_flags['client_name_lc'] = ods_name_flags['client_name'].fillna('').str.lower()

suspect_rows = ods_name_flags[
    ods_name_flags['client_name_lc'].str.contains(pattern_union, regex=True, na=False)
].copy()

suspect_inn = suspect_rows['inn'].dropna().unique().tolist()

suspect_summary = pd.DataFrame([{
    'suspect_rows_cnt': len(suspect_rows),
    'suspect_inn_cnt': len(suspect_inn),
    'share_inn_pct_from_ods_all': round(100.0 * len(suspect_inn) / max(ods_all_df['inn'].nunique(), 1), 2)
}])

display(suspect_summary)

print('Примеры строк ODS с подозрительным client_name (top 100):')
display(
    suspect_rows[['inn', 'client_name', 'contract_number', 'acq_class', 'd_valid_from', 'd_valid_to']]
    .drop_duplicates()
    .head(100)
)

excel_same_inn = excel_df[excel_df['inn'].isin(suspect_inn)].copy()

print('Сверка тех же ИНН в Excel (top 100):')
display(
    excel_same_inn[['inn', 'client_name', 'contract_number']]
    .drop_duplicates()
    .head(100)
)